# TDT Session Review

This notebook is used to review real TDT sessions saved by `src/tdt_host.py` in `data/sessions`.

The current experiment runs in the simplified threshold-first mode defined in `src/tdt_questplus.py` and executed by `src/tdt_host.py`:
- SOA range: `0-100 ms` in `1 ms` steps
- flash duration: `10 ms`
- practice block: `8` trials sampled without replacement from `{0, 15, 30, 45, 60, 75, 90, 105} ms`
- adaptive block: up to `64` trials
- sanity checks: inserted every `5` adaptive trials
- early stop: minimum `32` adaptive trials, `Threshold 50% CI95 width <= 5.0 ms`, and overall sanity accuracy `>= 50%`

The analysis cell loads the latest session (or a manually selected one), reads `summary.json` and `trials.csv`, and shows:
- an interactive plot of practice, main, and sanity trials
- the running threshold estimate with its CI range
- a text summary of the session
- a response table for the examiner

This notebook does not define the experiment logic itself; it is a viewer for completed sessions.


In [ ]:
import sys
from pathlib import Path


def find_source_root():
    candidates = (Path.cwd(), Path.cwd() / 'src', Path.cwd().parent)
    for candidate in candidates:
        if (candidate / 'tdt_questplus.py').exists():
            return candidate
    raise FileNotFoundError('Could not find tdt_questplus.py from the current notebook location.')


source_root = find_source_root()
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

import tdt_questplus as tdtq


config = tdtq.ExperimentConfig()
q = tdtq.build_staircase(config)

print('Notebook synced to the current threshold-first implementation in src/tdt_questplus.py')
print('')
print('Current configuration:')
print(f'  SOA range: {config.min_soa_ms}-{config.max_soa_ms} ms in {config.step_ms} ms steps')
print(f'  Flash duration: {config.flash_duration_ms} ms')
print(f'  Flash brightness: {config.flash_rgb_level}/255')
print(f'  Practice block: {config.practice_trial_count} trials -> {list(config.practice_soa_ms)}')
print(f'  Main adaptive block: up to {config.trial_count} trials')
print(f'  Response timeout: {config.response_timeout_ms} ms')
print(f'  Pre-stim delay: {config.prestim_delay_ms} ms')
print(f'  Inter-trial interval: {config.inter_trial_interval_ms} ms')
print(f'  Fixed sd values: {list(config.sd_values_ms)}')
print(f'  Fixed lapse values: {list(config.lapse_values)}')
print('')
print('Sanity checks:')
if config.sanity_checks.enabled:
    print(f'  Enabled: yes')
    print(f'  First inserted after adaptive trial: {config.sanity_checks.first_after_adaptive_trials}')
    print(f'  Then every adaptive trials: {config.sanity_checks.interval_adaptive_trials}')
    print(f'  Sequence: {list(config.sanity_checks.sequence)}')
    print(f'  0 ms control: {config.sanity_checks.zero_soa_ms} ms')
    print(
        '  High-interval control: '
        f"target ~{config.sanity_checks.easy_target_probability:.0%} 'Yes', "
        f"min {config.sanity_checks.easy_min_soa_ms} ms, max {config.sanity_checks.easy_max_soa_ms} ms, "
        'with an added uncertainty margin from the current CI width'
    )
    print(
        '  Early stop sanity gate: '
        f"overall sanity accuracy >= {config.stop_criteria.minimum_sanity_correct_rate:.0%}"
    )
else:
    print('  Enabled: no')
print('')

if config.stop_criteria.mean_entropy_normalized is None:
    print('Early stop:')
    print(f'  min trials >= {config.min_trials_for_early_stop}')
    print(f'  Threshold 50% CI95 width <= {config.stop_criteria.threshold50_ci95_width_ms:.1f} ms')
    print('  Entropy is report-only in the current version.')
    print(
        '  Overall sanity accuracy gate: '
        f"{config.stop_criteria.minimum_sanity_correct_rate:.0%}"
    )
else:
    print('Early stop:')
    print(f'  min trials >= {config.min_trials_for_early_stop}')
    print(f'  Threshold 50% CI95 width <= {config.stop_criteria.threshold50_ci95_width_ms:.1f} ms')
    print(f'  Normalized entropy <= {config.stop_criteria.mean_entropy_normalized:.3f}')

summary = tdtq.summarize_staircase(q, config, completed_trials=0)

print('')
print('Current prior summary:')
for line in tdtq.summary_lines(summary, config):
    print(f'  {line}')

print('')
print('Practice schedule used by the real host:')
print(f'  {tdtq.practice_schedule(config)}')

print('')
print("Set response_sequence = ['Yes', 'No', ...] below to simulate updates in the notebook.")

response_sequence = None

if response_sequence is None:
    print('')
    print('Notebook ready.')
else:
    completed_trials = 0
    simulation_summary = None

    for trial_number, response in enumerate(response_sequence[: config.trial_count], start=1):
        normalized_response = str(response).strip().capitalize()
        if normalized_response not in {'Yes', 'No'}:
            raise ValueError(f"Unsupported response: {response!r}. Use 'Yes' or 'No'.")

        next_stim = q.next_stim
        soa_ms = float(next_stim['intensity'])
        print(f'Trial {trial_number:02d}: interval={soa_ms:.0f} ms -> response={normalized_response}')
        q.update(stim=next_stim, outcome={'response': normalized_response})
        completed_trials = trial_number
        simulation_summary = tdtq.summarize_staircase(q, config, completed_trials)

        if simulation_summary['early_stop']['active']:
            print('')
            print(f'Early stop reached after {trial_number} trials.')
            break

    if simulation_summary is not None:
        print('')
        print('Final simulated summary:')
        for line in tdtq.summary_lines(simulation_summary, config):
            print(f'  {line}')


In [ ]:
import sys
from pathlib import Path

from IPython.display import HTML, display


def find_source_root():
    candidates = (Path.cwd(), Path.cwd() / 'src', Path.cwd().parent)
    for candidate in candidates:
        if (candidate / 'tdt_report.py').exists():
            return candidate
    raise FileNotFoundError('Could not find tdt_report.py from the current notebook location.')


source_root = find_source_root()
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

import tdt_report as tdtr


project_root = tdtr.find_project_root(source_root)
sessions_dir = project_root / 'data' / 'sessions'
session_dir_override = None  # e.g. sessions_dir / '20260323_131113_test_subject'
session_dir = session_dir_override or tdtr.latest_session_dir(project_root)

print(f'Loaded session: {session_dir.name}')
print('Rendered directly in the notebook output')

tdtr.export_report_html(session_dir)
display(HTML(tdtr.build_notebook_report_html(session_dir)))


Loaded session: 20260414_181145_test_subject
Rendered directly in the notebook output


Phase,Kind,Session #,Trial,Interval (ms),Lead LED,Flash level,Expected,Response,Correct,Button,RT (ms),T50 (ms),CI95 (ms)
practice,,1,1,30.0,flash1,255,,yes,None,blue,1149,,
practice,,2,2,90.0,flash1,255,,yes,None,blue,369,,
practice,,3,3,105.0,flash2,255,,yes,None,blue,275,,
practice,,4,4,75.0,flash1,255,,yes,None,blue,279,,
practice,,5,5,60.0,flash1,255,,yes,None,blue,273,,
practice,,6,6,0.0,simultaneous,255,,no,None,red,466,,
practice,,7,7,15.0,flash2,255,,yes,None,blue,392,,
practice,,8,8,45.0,flash2,255,,yes,None,blue,308,,
main,,9,1,50.0,flash1,255,,yes,None,blue,281,25.61,53.60
main,,10,2,25.0,flash1,255,,yes,None,blue,371,13.21,30.79
